sasho beshe tuk

In [3]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.colors as colors
import contextily as cx
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.metrics import classification_report
from sklearn.model_selection import RandomizedSearchCV

In [4]:
df = pd.read_parquet('./bird_habitats_weather.parquet')

### Sampling

In [ ]:
df.head()

In [ ]:
df.describe()

### Filtering the data based on date

In [ ]:
df_filtered = df[df['eventDate'].dt.year > 2019]

In [ ]:
df_filtered.head(5)

In [ ]:
df_filtered.dtypes

In [ ]:
df_filtered.tail(5)

### Creating the seen column

In [ ]:
# df_filtered['sasho'] = pd.to_numeric(pd.to_datetime(df_filtered['eventDate']))
# df_filtered['sasho1'] = pd.to_numeric(pd.to_datetime(df_filtered['date']))

In [ ]:
df_filtered['seen'] = (df_filtered['Phalacrocorax carbo'] > 0).astype(int)

### Splitting the data

In [ ]:
X = df_filtered.drop(columns=['Phalacrocorax carbo','seen'])

y = df_filtered['seen']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42)

In [ ]:
X_train.head(5)

In [ ]:
X_train.dtypes

In [ ]:
y_train.head(5)

### Baseline model

In [ ]:
from sklearn.ensemble import RandomForestClassifier

baseModel = RandomForestClassifier(
    random_state=42,
    n_jobs=-1,
    max_depth=15
)

X_train_numeric = X_train.drop(columns=['eventDate', 'main_habitat'], errors='ignore')
X_test_numeric = X_test.drop(columns=['eventDate', 'main_habitat'], errors='ignore')

baseModel.fit(X_train_numeric, y_train)
print(baseModel.predict(X_test_numeric))

In [ ]:
import sklearn.metrics as skm
from sklearn.metrics import roc_curve, auc

In [ ]:
acc = baseModel.score(X_test_numeric,y_test)

Y_score = baseModel.predict_proba(X_test_numeric)[:,1]
fpr = dict()
tpr = dict()
fpr, tpr, sasho  = roc_curve(y_test, Y_score)

roc_auc = dict()
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(10,10))
plt.plot([0, 1], [0, 1], 'k--')
plt.xlim([-0.05, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.grid(True)
plt.plot(fpr, tpr, label='AUC = {0}'.format(roc_auc))
plt.legend(loc="lower right", shadow=True, fancybox =True) 
plt.show()

In [ ]:
print(acc)

In [ ]:
y_pred = baseModel.predict(X_test_numeric)

report = classification_report(y_test, y_pred)

In [ ]:
print(report)

hyperparameter tuning

In [ ]:
clf = RandomForestClassifier(random_state=0)
param_ranges = {
    'n_estimators': [100, 150, 200, 300],
    'max_depth': [8, 12, 15, 20, None],
    'min_samples_split': [2, 5, 10, 20],
    'min_samples_leaf': [5, 10, 20, 50],
    'max_features': ["sqrt", "log2", None],
    'class_weight': ["balanced", "balanced_subsample"]
}

tuner = RandomizedSearchCV(
    estimator=clf, 
    param_distributions=param_ranges, 
    n_iter=10, 
    cv=3,              
    random_state=42,
    n_jobs=-1          
)

tuner.fit(X_train, y_train)
best_clf = tuner.best_estimator_
print(f"Best settings found: {tuner.best_params_}\n")

train_pred = best_clf.predict(X_train)
test_pred = best_clf.predict(X_test)

print(f'Tuned Train accuracy: {accuracy_score(y_train, train_pred)}') 
print(f'Tuned Test accuracy: {accuracy_score(y_test, test_pred)}')

crazy burger model

In [ ]:
def burger_model(row): 
    return 0

In [ ]:
df_eval = df.copy()
df_eval['predicted_seen'] = df_eval.apply(lambda row: burger_model(row), axis=1)

y_true = df_eval["seen"]
y_pred = df_eval["predicted_seen"]

In [ ]:
y_pred = burger_model.predict(X_test_numeric)

report = classification_report(y_test, y_pred)

In [ ]:
print(report)